# 05. Other classifiers (logistic regression and random forest)

Parul. Alternatives to the SVM in `04_classifier_svm.ipynb`, using the same combined 8206-d features (and a PCA-95% variant).


In [ ]:
# Colab setup
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install numpy pandas matplotlib scikit-learn joblib
    if not os.path.exists("/content/MIDS-281-project/.git"):
        !git clone --depth 1 https://github.com/sreeramchennakrish/MIDS-281-project.git /content/MIDS-281-project
    %cd /content/MIDS-281-project/notebooks

print("cwd:", os.getcwd())


## Load combined features


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

FEATURE_DIR = Path("../features").resolve()
OUTPUT_DIR = Path("../outputs/dimred").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with np.load(FEATURE_DIR / "combined_weighted_features.npz") as data:
    X_train = data["X_train"]
    X_val = data["X_val"]
    y_train = data["y_train"]
    y_val = data["y_val"]
    classes = np.asarray(data["classes"])
    feature_groups = data["feature_groups"]
    feature_dimensions = data["feature_dimensions"]
    feature_weights = data["feature_weights"]

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Classes:", classes)
for name, dim, weight in zip(feature_groups, feature_dimensions, feature_weights):
    print(f"{name}: {dim} features, weight={weight}")


## Train and evaluate


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay, precision_score, recall_score,
)

MODEL_DIR = Path("../models/other_classifiers").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def evaluate(name, model, Xtr, ytr, Xva, yva):
    model.fit(Xtr, ytr)
    pred = model.predict(Xva)
    print("=" * 60)
    print(name)
    print("Accuracy:", accuracy_score(yva, pred))
    print(classification_report(yva, pred, target_names=classes))
    ConfusionMatrixDisplay.from_predictions(yva, pred, display_labels=classes, cmap="Blues")
    plt.title(name)
    plt.show()
    return {
        "Model": name,
        "Accuracy": accuracy_score(yva, pred),
        "Precision": precision_score(yva, pred, average="macro"),
        "Recall": recall_score(yva, pred, average="macro"),
        "F1 Score": f1_score(yva, pred, average="macro"),
        "pred": pred,
        "fitted": model,
    }


In [ ]:
results = []

logreg = Pipeline([
    ("clf", LogisticRegression(max_iter=2000, multi_class="auto", random_state=42)),
])
results.append(evaluate("Logistic regression (combined 8206-d)", logreg, X_train, y_train, X_val, y_val))

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)
results.append(evaluate("Random forest (combined 8206-d)", rf, X_train, y_train, X_val, y_val))

# Same models on PCA-95% features (fit PCA on train only)
pca95 = PCA(n_components=0.95, random_state=42)
Xtr_p = pca95.fit_transform(X_train)
Xva_p = pca95.transform(X_val)
print("PCA 95% components:", Xtr_p.shape[1])

logreg_pca = LogisticRegression(max_iter=2000, random_state=42)
results.append(evaluate(f"Logistic regression (PCA {Xtr_p.shape[1]}-d)", logreg_pca, Xtr_p, y_train, Xva_p, y_val))

rf_pca = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42
)
results.append(evaluate(f"Random forest (PCA {Xtr_p.shape[1]}-d)", rf_pca, Xtr_p, y_train, Xva_p, y_val))


## Comparison


In [ ]:
rows = [{k: v for k, v in r.items() if k not in ("pred", "fitted")} for r in results]
comparison_df = pd.DataFrame(rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
display(comparison_df.round(4))

import joblib
joblib.dump(
    {r["Model"]: r["fitted"] for r in results},
    MODEL_DIR / "logreg_rf_bundle.joblib",
)
print("Saved models to", MODEL_DIR)
print("SVM remains the team baseline in 04_classifier_svm.ipynb. This notebook is the non-SVM comparison.")
